# EE2211 Tutorial 10 — Question 6
## 5-Fold Cross-Validation: Polynomial Order Selection on Iris

**Problem (Q6):** Load `sklearn.datasets.load_iris`. Perform 5-fold cross-validation to find the
best polynomial order (1–10, no regularization) for classification. Partition the dataset into
training / validation / test, where **val set size = test set size**. Plot average 5-fold training
and validation error vs polynomial order. Maintain the randomly partitioned folds for reuse.
Investigate the "Singular Matrix" error at high orders, and whether ridge regularization helps.

### Data split design
- Total: 150 samples, 4 features, 3 classes
- Test: **25 samples** (held out, not used in CV)
- CV pool: **125 samples** → 5 equal folds of **25** each
- Per fold: train = 100, val = 25 → val size = test size ✓

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from numpy.linalg import inv

## Step 1 — Load Iris, OHE targets, hold out test set

In [ ]:
iris = load_iris()
X_all = iris.data          # (150, 4)
y_all = iris.target        # (150,)  integers 0/1/2

# OHE encode class labels → regression targets
ohe = OneHotEncoder(sparse_output=False)
Y_all = ohe.fit_transform(y_all.reshape(-1, 1))  # (150, 3)

# Reproducible random permutation
np.random.seed(42)
perm = np.random.permutation(150)

# Hold out 25 as permanent test set
test_idx = perm[:25]
cv_idx   = perm[25:]       # 125 samples for CV

X_test, Y_test = X_all[test_idx], Y_all[test_idx]
X_cv,   Y_cv   = X_all[cv_idx],   Y_all[cv_idx]

print(f'Test set : {X_test.shape}   CV pool : {X_cv.shape}')

## Step 2 — Build 5 CV folds (saved for reuse)

In [ ]:
def make_folds(n, k=5):
    """Return list of (train_indices, val_indices) for k-fold CV over n samples."""
    fold_size = n // k
    folds = []
    for f in range(k):
        val_start = f * fold_size
        val_end   = (f + 1) * fold_size if f < k - 1 else n
        val_idx   = np.arange(val_start, val_end)
        train_idx = np.concatenate([np.arange(0, val_start), np.arange(val_end, n)])
        folds.append((train_idx, val_idx))
    return folds

folds = make_folds(len(X_cv), k=5)   # kept for reuse in future algorithms

for i, (tr, va) in enumerate(folds):
    print(f'Fold {i+1}: train={len(tr)}, val={len(va)}')

## Step 3 — Polynomial sweep, orders 1–10, no regularization

In [ ]:
def polyTx(X, order):
    return PolynomialFeatures(degree=order, include_bias=True).fit_transform(X)

def solvePR(P, Y, lam=0.0):
    PtP = P.T @ P
    reg = lam * np.eye(PtP.shape[0])
    return inv(PtP + reg) @ P.T @ Y

def error_rate(P, W, Y_ohe):
    pred  = np.argmax(P @ W, axis=1)
    true  = np.argmax(Y_ohe, axis=1)
    return np.mean(pred != true)

orders = list(range(1, 11))
train_errs, val_errs = [], []

for order in orders:
    fold_tr, fold_va = [], []
    for tr_idx, va_idx in folds:
        X_tr, Y_tr = X_cv[tr_idx], Y_cv[tr_idx]
        X_va, Y_va = X_cv[va_idx], Y_cv[va_idx]
        P_tr = polyTx(X_tr, order)
        P_va = polyTx(X_va, order)
        try:
            W = solvePR(P_tr, Y_tr, lam=0.0)
            fold_tr.append(error_rate(P_tr, W, Y_tr))
            fold_va.append(error_rate(P_va, W, Y_va))
        except np.linalg.LinAlgError as e:
            print(f'  Order {order}: Singular Matrix — {e}')
            fold_tr.append(np.nan)
            fold_va.append(np.nan)
    train_errs.append(np.nanmean(fold_tr))
    val_errs.append(np.nanmean(fold_va))

best_order = orders[np.nanargmin(val_errs)]
print(f'\nBest polynomial order (min val error): {best_order}')
print(f'Orders : {orders}')
print(f'Train  : {[round(e,3) for e in train_errs]}')
print(f'Val    : {[round(e,3) for e in val_errs]}')

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(orders, train_errs, 'b-o', label='Train error (no reg)')
plt.plot(orders, val_errs,   'r-o', label='Val error (no reg)')
plt.xlabel('Polynomial order')
plt.ylabel('Error rate')
plt.title('5-Fold CV — Training vs Validation Error Rate')
plt.xticks(orders)
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Discussion — Singular Matrix at High Orders

**Why does it happen?**  
Polynomial expansion of a 4-feature input at order $p$ creates $\binom{4+p}{p}$ columns in $P$.
At high orders, these columns contain products like $x_1^2 x_2$ and $x_1 x_2^2$ that become nearly
linearly dependent (especially when feature values span a narrow range). This makes $P^\top P$
singular (or near-singular), so `inv(P'P)` fails.

**Number of features** at each order:  
order 1 → 5, order 2 → 15, order 3 → 35, order 4 → 70, order 5 → 126 (> 100 training points!)

When #features > #training samples, the system is **under-determined** and $P^\top P$ is always
singular.

**Does ridge regularization fix it?**  
Yes — adding $\lambda I$ to $P^\top P$ guarantees the matrix is invertible for any $\lambda > 0$.
**Trade-off**: a non-zero $\lambda$ increases bias, so the learned $w^*$ is no longer the unregularized
global minimum. However, in practice it substantially reduces overfitting, so validation error
often improves.

In [ ]:
# Repeat sweep with ridge regularization (λ=1e-3)
lam = 1e-3
train_errs_reg, val_errs_reg = [], []

for order in orders:
    fold_tr, fold_va = [], []
    for tr_idx, va_idx in folds:
        X_tr, Y_tr = X_cv[tr_idx], Y_cv[tr_idx]
        X_va, Y_va = X_cv[va_idx], Y_cv[va_idx]
        P_tr = polyTx(X_tr, order)
        P_va = polyTx(X_va, order)
        W = solvePR(P_tr, Y_tr, lam=lam)
        fold_tr.append(error_rate(P_tr, W, Y_tr))
        fold_va.append(error_rate(P_va, W, Y_va))
    train_errs_reg.append(np.mean(fold_tr))
    val_errs_reg.append(np.mean(fold_va))

best_order_reg = orders[np.argmin(val_errs_reg)]
print(f'Best order with ridge (λ={lam}): {best_order_reg}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(orders, train_errs, 'b-o', label='Train')
axes[0].plot(orders, val_errs,   'r-o', label='Val')
axes[0].set_title('No Regularization')
axes[0].set_xlabel('Polynomial order'); axes[0].set_ylabel('Error rate')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(orders, train_errs_reg, 'b-o', label='Train')
axes[1].plot(orders, val_errs_reg,   'r-o', label='Val')
axes[1].set_title(f'Ridge λ={lam}')
axes[1].set_xlabel('Polynomial order'); axes[1].set_ylabel('Error rate')
axes[1].legend(); axes[1].grid(True)

plt.suptitle('5-Fold CV: Effect of Regularization on Polynomial Order Selection')
plt.tight_layout()
plt.show()

In [ ]:
# Final evaluation on held-out test set using best order from CV
# Train on the FULL CV pool (125 samples)
P_cv_full  = polyTx(X_cv,  best_order_reg)
P_test_full = polyTx(X_test, best_order_reg)
W_final = solvePR(P_cv_full, Y_cv, lam=lam)

test_err = error_rate(P_test_full, W_final, Y_test)
print(f'Best order (ridge CV): {best_order_reg}')
print(f'Test error rate      : {test_err:.3f}  ({(1-test_err)*100:.1f}% accuracy)')